# End-to-End GNN Pipeline (Graph Construction Phase)

This notebook demonstrates the **Graph Construction** phase of the GNN-based AML detection system.
It covers:
1.  Data Loading (Mock Data Generation)
2.  Preprocessing & ID Mapping
3.  Heterogeneous Graph Construction (`HeteroData`)
4.  Graph Inspection

In [ ]:
import pandas as pd
import numpy as np
import torch
from src.gnn.graph.builder import GraphBuilder
from src.gnn.config.schema import Schema

print("Libraries loaded successfully.")

## 1. Data Generation
Since we are in a development environment, we generate a synthetic dataset mimicking the PayPal transaction schema.

In [ ]:
# Generate Synthetic Data
def generate_mock_data(num_txns=100):
    data = {
        Schema.TRANSACTION_ID: [f'txn_{i}' for i in range(num_txns)],
        Schema.DATE: pd.date_range(start='2023-01-01', periods=num_txns, freq='H'),
        Schema.VOLUME: np.random.uniform(10, 1000, num_txns),
        Schema.DIRECTION: np.random.choice(['Inbound', 'Outbound'], num_txns),
        Schema.CUSTOMER_NAME: np.random.choice(['Customer_A', 'Customer_B', 'Customer_C', 'Customer_D', 'Supernode_X'], num_txns),
        Schema.BANK_NAME: np.random.choice(['Bank_Alpha', 'Bank_Beta', 'Bank_Gamma'], num_txns),
        # Accounts: Some shared, some unique
        Schema.BANK_ACCOUNT: np.random.choice([f'Acc_{i}' for i in range(10)], num_txns),
        # Features that might be pre-calculated
        Schema.RARITY_SCORE: np.random.uniform(0, 1, num_txns),
        Schema.TIMESTAMP_NORM: np.linspace(0, 1, num_txns)
    }
    return pd.DataFrame(data)

df = generate_mock_data(200)
print(f"Generated {len(df)} transactions.")
df.head()

## 2. Graph Construction
We use the `GraphBuilder` class to transform the tabular data into a PyG `HeteroData` object.
This handles:
1.  Mapping string IDs (Customers, Accounts) to integer indices.
2.  Creating edge indices (Customer -> Account, Account -> Transaction).
3.  Aggregating/Assigning node features.

In [ ]:
# Initialize Builder
builder = GraphBuilder()

# Build Graph
graph = builder.build(df)

print("Graph construction complete.")
print(graph)

## 3. Graph Inspection
Let's examine the structure of the constructed graph.

In [ ]:
# Node Types and Counts
for node_type in graph.node_types:
    print(f"Node Type: {node_type}, Count: {graph[node_type].num_nodes}")

# Edge Types and Connections
for edge_type in graph.edge_types:
    src, rel, dst = edge_type
    count = graph[edge_type].edge_index.shape[1]
    print(f"Edge: ({src} -[{rel}]-> {dst}), Count: {count}")

In [ ]:
# Feature Inspection
print("\n--- Feature Shapes ---")
print(f"Customer Features (x): {graph[Schema.NODE_CUSTOMER].x.shape}  [rarity, volume, count]")
print(f"Account Features (x):  {graph[Schema.NODE_ACCOUNT].x.shape}   [embedding_placeholder]")
print(f"Transaction Features (x): {graph[Schema.NODE_TRANSACTION].x.shape} [volume, time, dir]")

## 4. Next Steps
The `HeteroData` object `graph` is now ready to be fed into a GNN model (e.g., GAT) for training.
The next phase involves:
1.  Defining the `FraudGAT` model architecture.
2.  Implementing the training loop (Negative Sampling + Link Prediction loss).